# HW01-B — SQL, Latency, and Metabase

The business team does not care that your notebook works. They want a dashboard that opens fast.

Here you connect to shared Postgres, write SQL, measure latency, create a materialized view in your own schema, and build a Metabase dashboard.

## Submission discipline

This is individual work.

Work locally. Push to GitHub. Use the shared server services through URLs and credentials. Do not SSH into the server.

Do not commit `.env`, `.venv/`, passwords.

## Credentials and shared services

Credentials, service URLs, and connection details are provided on the HW page.

Use those exact values. Everyone must work against the same QBC12 database snapshot and the same shared Metabase/Airflow services.

Do not paste credentials into notebook markdown. Do not commit `.env` files. Do not screenshot passwords.


## Useful references

- PostgreSQL `EXPLAIN`: https://www.postgresql.org/docs/current/sql-explain.html
- PostgreSQL using `EXPLAIN`: https://www.postgresql.org/docs/current/using-explain.html
- Metabase questions: https://www.metabase.com/docs/latest/questions/introduction
- Metabase dashboards: https://www.metabase.com/docs/latest/dashboards/introduction

if you cannot open any one of these contact me : Bale (arianaghamohseni, image of a scared chicken), or Telegram (@arianaghamohseni)

## What to avoid

- `select *` in dashboard queries.
- Creating objects in `core`. You do not own `core`.
- Optimizing without runtime measurements.
- Making Metabase run a massive join every time someone opens the dashboard.

In [1]:
from dotenv import load_dotenv
load_dotenv()

True

In [5]:
import os, re, time
from pathlib import Path
import pandas as pd
from sqlalchemy import create_engine, text

for path in ['sql', 'reports', 'screenshots']:
    Path(path).mkdir(exist_ok=True)

DB_HOST = os.getenv('QBC12_DB_HOST', 'SERVERIP')    # this is in the excel file give in Quera
DB_PORT = os.getenv('QBC12_DB_PORT', '32112')
DB_NAME = os.getenv('QBC12_DB_NAME', 'qbc12_airbnb')
DB_USER = os.getenv('QBC12_DB_USER', '') or input('DB user: ').strip()
DB_PASSWORD = os.getenv('QBC12_DB_PASSWORD', '') or input('DB password: ').strip()
STUDENT_ID = os.getenv('QBC12_STUDENT_ID', '') or DB_USER.replace('student_', '')

safe_student = re.sub(r'[^a-zA-Z0-9_]', '_', STUDENT_ID.lower())
STUDENT_SCHEMA = f'student_{safe_student}'
engine = create_engine(f'postgresql+psycopg2://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}', pool_pre_ping=True)
with engine.begin() as conn:
    conn.execute(text("SET statement_timeout = '30s'"))
    version = conn.execute(text('select version()')).scalar()
STUDENT_SCHEMA, version[:80]

OperationalError: (psycopg2.OperationalError) connection to server at "185.50.38.163", port 32112 failed: Connection timed out (0x0000274C/10060)
	Is the server running on that host and accepting TCP/IP connections?

(Background on this error at: https://sqlalche.me/e/20/e3q8)

## 1. Inspect before querying

You are not allowed to write the final query blind. Check columns and row counts first.

In [3]:
columns_sql = '''
select table_schema, table_name, column_name, data_type
from information_schema.columns
where table_schema = 'core'
  and table_name in ('listing', 'calendar_day', 'review')
order by table_name, ordinal_position;
'''
pd.read_sql(columns_sql, engine)

,table_schema,table_name,column_name,data_type
0,core,calendar_day,listing_id,bigint
1,core,calendar_day,date,date
2,core,calendar_day,available,boolean
3,core,calendar_day,price,numeric
4,core,calendar_day,adjusted_price,numeric
5,core,calendar_day,minimum_nights,integer
6,core,calendar_day,maximum_nights,integer
7,core,listing,listing_id,bigint
8,core,listing,host_id,bigint
9,core,listing,neighbourhood_id,integer


In [4]:
row_count_sql = '''
select 'core.listing' as table_name, count(*) as rows from core.listing
union all select 'core.calendar_day', count(*) from core.calendar_day
union all select 'core.review', count(*) from core.review;
'''
pd.read_sql(row_count_sql, engine)

,table_name,rows
0,core.listing,10480
1,core.calendar_day,3825200
2,core.review,501084


## 2. Create your sandbox schema

This is the only place you write database objects.

In [5]:
# TODO 2.1
# Create your schema if it does not exist.
# Schema name is STUDENT_SCHEMA.

# Write your code here.
with engine.begin() as conn:
    conn.execute(text(f'CREATE SCHEMA IF NOT EXISTS "{STUDENT_SCHEMA}"'))
print(f"Schema '{STUDENT_SCHEMA}' is ready.")

ProgrammingError: (psycopg2.errors.InsufficientPrivilege) permission denied for database qbc12_airbnb

[SQL: CREATE SCHEMA IF NOT EXISTS "student_0372"]
(Background on this error at: https://sqlalche.me/e/20/f405)

Based on the result of the previous cell, I found that I don't have permission to create new schema, so I checked that maybe the schema has created already but with different name:

In [6]:
# Check what schemas you CAN see and use
available_schemas_sql = '''
SELECT schema_name 
FROM information_schema.schemata
ORDER BY schema_name;
'''
pd.read_sql(available_schemas_sql, engine)

,schema_name
0,core
1,information_schema
2,pg_catalog
3,public
4,student_melika_nobakhtian


Because I saw that there is a schema with my name but with different format, I changed `STUDENT_SCHEMA` to make things work:

In [7]:
STUDENT_SCHEMA = 'student_melika_nobakhtian'
print(f"Using schema: {STUDENT_SCHEMA}")

Using schema: student_melika_nobakhtian


## 3. Build baseline SQL in pieces

Do not write one giant query first. Build the CTEs, test them, then combine.

I checked and saw that `price` is NULL in `calendar_day`:

In [8]:
# Check available vs unavailable price distribution
pd.read_sql('''
SELECT 
    available,
    COUNT(*) as rows,
    COUNT(price) as non_null_price,
    COUNT(*) - COUNT(price) as null_price
FROM core.calendar_day
WHERE date >= CURRENT_DATE
  AND date < CURRENT_DATE + INTERVAL '30 days'
GROUP BY available
''', engine)

,available,rows,non_null_price,null_price
0,False,240120,0,240120
1,True,74280,0,74280


But `price` exists in the `listing`, so we should join tables:

In [9]:
# Check sample data to understand the situation
pd.read_sql('''
SELECT 
    c.listing_id,
    c.available,
    c.price,
    c.adjusted_price,
    l.listing_price
FROM core.calendar_day c
JOIN core.listing l ON l.listing_id = c.listing_id
LIMIT 10
''', engine)

,listing_id,available,price,adjusted_price,listing_price
0,792781586502433878,False,None,None,314.0
1,792781586502433878,False,None,None,314.0
2,792781586502433878,False,None,None,314.0
3,792781586502433878,False,None,None,314.0
4,792781586502433878,False,None,None,314.0
5,792781586502433878,False,None,None,314.0
6,792781586502433878,False,None,None,314.0
7,792781586502433878,False,None,None,314.0
8,792781586502433878,False,None,None,314.0
9,792781586502433878,False,None,None,314.0


I added `COALESCE` to handle Nan values in `avg_calendar_price_30` column that are here becuase of LEFT JOIN:

In [10]:
# TODO 3.1
# Write calendar_30_sql.
# Required output: listing_id, avg_calendar_price_30, availability_30_rate.

calendar_30_sql = '''
SELECT
    c.listing_id,
    ROUND(AVG(COALESCE(l.listing_price, 0))::numeric, 2)                AS avg_calendar_price_30,
    ROUND(AVG(CASE WHEN c.available THEN 1.0 ELSE 0.0 END)::numeric, 4) AS availability_30_rate
FROM core.calendar_day c
LEFT JOIN core.listing l ON l.listing_id = c.listing_id
WHERE c.date >= CURRENT_DATE
  AND c.date < CURRENT_DATE + INTERVAL '30 days'
GROUP BY c.listing_id
'''
pd.read_sql(calendar_30_sql, engine).head(10)

,listing_id,avg_calendar_price_30,availability_30_rate
0,1443670960781261954,146.0,1.0
1,896043282611946316,0.0,0.0
2,39969190,0.0,0.0
3,958726726744532841,0.0,0.0
4,1272264495001498383,125.0,1.0
5,1093563123501570178,133.0,0.0
6,1476051889347548382,214.0,1.0
7,21084288,0.0,0.0
8,1041257783578352876,0.0,0.0
9,14572858,256.0,0.0


In [11]:
# TODO 3.2
# Write review_counts_sql.
# Required output: listing_id, total_reviews.

review_counts_sql = '''
SELECT
    listing_id,
    COUNT(*) AS total_reviews
FROM core.review
GROUP BY listing_id
'''

pd.read_sql(review_counts_sql, engine).head()

,listing_id,total_reviews
0,1443670960781261954,5
1,896043282611946316,2
2,958726726744532841,23
3,39969190,10
4,1476051889347548382,2


We already computed `availability_30_rate` per listing in the CTE. But in the main query we're grouping by `neighbourhood`, so one neighbourhood has many listings, each with their own rate. The AVG takes the average of all listing rates within that neighbourhood:

In [16]:
# TODO 3.3
# Combine the CTEs with core.listing into baseline_sql.
# Required output:
# neighbourhood, num_listings, avg_price, median_price,
# avg_minimum_nights, total_reviews, reviews_per_listing, availability_30_rate.

baseline_sql = '''
WITH calendar_30 AS (
    SELECT
        listing_id,
        ROUND(AVG(CASE WHEN available THEN 1.0 ELSE 0.0 END)::numeric, 4) AS availability_30_rate
    FROM core.calendar_day
    WHERE date >= CURRENT_DATE
      AND date < CURRENT_DATE + INTERVAL '30 days'
    GROUP BY listing_id
),
review_counts AS (
    SELECT
        listing_id,
        COUNT(*) AS total_reviews
    FROM core.review
    GROUP BY listing_id
)
SELECT
    l.neighbourhood_id::text                                                               AS neighbourhood,
    COUNT(l.listing_id)                                                                    AS num_listings,
    ROUND(AVG(l.listing_price)::numeric, 2)                                                AS avg_price,
    ROUND(PERCENTILE_CONT(0.5) WITHIN GROUP (ORDER BY l.listing_price)::numeric, 2)       AS median_price,
    ROUND(AVG(l.minimum_nights)::numeric, 2)                                               AS avg_minimum_nights,
    COALESCE(SUM(r.total_reviews), 0)                                                      AS total_reviews,
    ROUND(COALESCE(SUM(r.total_reviews)::numeric / NULLIF(COUNT(l.listing_id), 0), 0), 2) AS reviews_per_listing,
    ROUND(AVG(c30.availability_30_rate)::numeric, 4)                                       AS availability_30_rate
FROM core.listing l
LEFT JOIN calendar_30   c30 ON c30.listing_id = l.listing_id
LEFT JOIN review_counts r   ON r.listing_id   = l.listing_id
GROUP BY l.neighbourhood_id
'''
Path('sql/01_baseline_neighbourhood_summary.sql').write_text(baseline_sql)

1477

In [17]:
def timed_read_sql(sql: str, repeats: int = 3):
    times = []
    last_df = None
    for _ in range(repeats):
        start = time.perf_counter()
        last_df = pd.read_sql(sql, engine)
        times.append(time.perf_counter() - start)
    return last_df, times

baseline_df, baseline_times = timed_read_sql(baseline_sql, repeats=3)
baseline_df.head(), baseline_times

(  neighbourhood  num_listings  avg_price  median_price  avg_minimum_nights  \
 0             1           923     307.72         240.0                4.89   
 1             2          1207     315.88         245.5                4.01   
 2             3           436     216.60         195.0                4.40   
 3             4           736     255.04         214.5                4.24   
 4             5           206     193.24         185.5                3.36   
 
    total_reviews  reviews_per_listing  availability_30_rate  
 0        76899.0                83.31                0.3449  
 1       106496.0                88.23                0.2904  
 2        13988.0                32.08                0.2040  
 3        26668.0                36.23                0.2087  
 4         8725.0                42.35                0.1723  ,
 [0.5616008999932092, 0.5556423000089126, 0.6013269000104629])

## 4. Read the query plan

`EXPLAIN ANALYZE` actually runs the query. Look for big scans, expensive joins, and repeated work.

In [18]:
# TODO 4.1
# Run EXPLAIN (ANALYZE, BUFFERS, FORMAT TEXT) on baseline_sql.
# Save the plan to reports/baseline_explain_analyze.txt.

# Write your code here.
explain_sql = f"EXPLAIN (ANALYZE, BUFFERS, FORMAT TEXT) {baseline_sql}"

with engine.connect() as conn:
    conn.execute(text("SET statement_timeout = '120s'"))
    rows = conn.execute(text(explain_sql)).fetchall()

plan_text = "\n".join(row[0] for row in rows)
Path('reports/baseline_explain_analyze.txt').write_text(plan_text)
print(plan_text[:3000])   # preview first 3000 chars

GroupAggregate  (cost=50659.99..50897.48 rows=22 width=236) (actual time=270.603..276.905 rows=22 loops=1)
  Group Key: l.neighbourhood_id
  Buffers: shared hit=15081 read=6009
  ->  Sort  (cost=50659.99..50686.26 rows=10506 width=61) (actual time=269.978..271.459 rows=10480 loops=1)
        Sort Key: l.neighbourhood_id
        Sort Method: quicksort  Memory: 1071kB
        Buffers: shared hit=15081 read=6009
        ->  Hash Left Join  (cost=49640.42..49958.25 rows=10506 width=61) (actual time=249.176..266.696 rows=10480 loops=1)
              Hash Cond: (l.listing_id = r.listing_id)
              Buffers: shared hit=15081 read=6009
              ->  Hash Right Join  (cost=35913.49..36203.73 rows=10506 width=53) (actual time=158.781..173.265 rows=10480 loops=1)
                    Hash Cond: (calendar_day.listing_id = l.listing_id)
                    Buffers: shared hit=12874
                    ->  Finalize HashAggregate  (cost=35487.69..35645.28 rows=10506 width=40) (actual time=14

In [19]:
# TODO 4.2
# Write reports/explain_notes.md with 3 specific observations from the plan.
# Do not write vague nonsense like 'the query is slow'.

# Write your code here.
notes = """# Baseline Query Plan — Observations

## Observation 1 — Parallel Bitmap Heap Scan reads 314,400 rows across 3 workers
Despite an index on date (idx_calendar_date), Postgres still visits 5,296 heap blocks
to fetch actual row data for 104,800 rows per worker across 3 parallel workers.
Total buffer usage for this node: shared hit=12,684 — 87% of all buffer hits in the plan.

## Observation 2 — Two chained Hash Joins rebuild 4,241kB hash table every execution
A Finalize HashAggregate builds a 4,241kB hash table for 10,480 rows (actual time=147..157ms),
which feeds into a Hash Right Join with core.listing, then a Hash Left Join with review_counts.
This entire chain is rebuilt from scratch on every single query execution with no reuse.

## Observation 3 — 1,071kB sort on 10,480 rows to produce just 22 output groups
Postgres sorts all 10,480 joined rows by neighbourhood_id (1,071kB quicksort)
before GroupAggregate collapses them into 22 final rows (276ms total end-to-end).
99.8% of processed rows are intermediate — a materialized view eliminates this entirely.
"""

Path('reports/explain_notes.md').write_text(notes)
print(notes)

# Baseline Query Plan — Observations

## Observation 1 — Parallel Bitmap Heap Scan reads 314,400 rows across 3 workers
Despite an index on date (idx_calendar_date), Postgres still visits 5,296 heap blocks
to fetch actual row data for 104,800 rows per worker across 3 parallel workers.
Total buffer usage for this node: shared hit=12,684 — 87% of all buffer hits in the plan.

## Observation 2 — Two chained Hash Joins rebuild 4,241kB hash table every execution
A Finalize HashAggregate builds a 4,241kB hash table for 10,480 rows (actual time=147..157ms),
which feeds into a Hash Right Join with core.listing, then a Hash Left Join with review_counts.
This entire chain is rebuilt from scratch on every single query execution with no reuse.

## Observation 3 — 1,071kB sort on 10,480 rows to produce just 22 output groups
Postgres sorts all 10,480 joined rows by neighbourhood_id (1,071kB quicksort)
before GroupAggregate collapses them into 22 final rows (276ms total end-to-end).
99.8% of processed

## 5. Create a materialized view

Metabase should read from a prepared object, not a fresh monster join.

In [20]:
# TODO 5.1
# Create optimized_sql.
# It should create student_<you>.mv_airbnb_neighbourhood_summary and at least two indexes.

optimized_sql = f'''
DROP MATERIALIZED VIEW IF EXISTS "{STUDENT_SCHEMA}".mv_airbnb_neighbourhood_summary;

CREATE MATERIALIZED VIEW "{STUDENT_SCHEMA}".mv_airbnb_neighbourhood_summary AS
WITH calendar_30 AS (
    SELECT
        listing_id,
        ROUND(AVG(CASE WHEN available THEN 1.0 ELSE 0.0 END)::numeric, 4) AS availability_30_rate
    FROM core.calendar_day
    WHERE date >= CURRENT_DATE
      AND date < CURRENT_DATE + INTERVAL '30 days'
    GROUP BY listing_id
),
review_counts AS (
    SELECT
        listing_id,
        COUNT(*) AS total_reviews
    FROM core.review
    GROUP BY listing_id
)
SELECT
    l.neighbourhood_id::text                                                               AS neighbourhood,
    COUNT(l.listing_id)                                                                    AS num_listings,
    ROUND(AVG(l.listing_price)::numeric, 2)                                                AS avg_price,
    ROUND(PERCENTILE_CONT(0.5) WITHIN GROUP (ORDER BY l.listing_price)::numeric, 2)       AS median_price,
    ROUND(AVG(l.minimum_nights)::numeric, 2)                                               AS avg_minimum_nights,
    COALESCE(SUM(r.total_reviews), 0)                                                      AS total_reviews,
    ROUND(COALESCE(SUM(r.total_reviews)::numeric / NULLIF(COUNT(l.listing_id), 0), 0), 2) AS reviews_per_listing,
    ROUND(AVG(c30.availability_30_rate)::numeric, 4)                                       AS availability_30_rate
FROM core.listing l
LEFT JOIN calendar_30   c30 ON c30.listing_id = l.listing_id
LEFT JOIN review_counts r   ON r.listing_id   = l.listing_id
GROUP BY l.neighbourhood_id;

-- Index 1: fast lookup and filtering by neighbourhood
CREATE INDEX idx_mv_neighbourhood
    ON "{STUDENT_SCHEMA}".mv_airbnb_neighbourhood_summary (neighbourhood);

-- Index 2: fast sorting by number of listings ]
CREATE INDEX idx_mv_num_listings
    ON "{STUDENT_SCHEMA}".mv_airbnb_neighbourhood_summary (num_listings DESC);
'''

Path('sql/02_create_materialized_view.sql').write_text(optimized_sql)

2006

In [21]:
# TODO 5.2
# Execute optimized_sql statement by statement.

# Write your code here.
statements = [s.strip() for s in optimized_sql.split(';') if s.strip()]

with engine.begin() as conn:
    conn.execute(text("SET statement_timeout = '300s'"))
    for stmt in statements:
        print(f"Running: {stmt[:80]}...")
        conn.execute(text(stmt))

print("Done — materialized view and indexes created.")

Running: DROP MATERIALIZED VIEW IF EXISTS "student_melika_nobakhtian".mv_airbnb_neighbour...
Running: CREATE MATERIALIZED VIEW "student_melika_nobakhtian".mv_airbnb_neighbourhood_sum...
Running: -- Index 1: fast lookup and filtering by neighbourhood
CREATE INDEX idx_mv_neigh...
Running: -- Index 2: fast sorting by number of listings ]
CREATE INDEX idx_mv_num_listing...
Done — materialized view and indexes created.


In [23]:
check_sql = f'''select * from "{STUDENT_SCHEMA}".mv_airbnb_neighbourhood_summary order by num_listings desc limit 10;'''
pd.read_sql(check_sql, engine)

OperationalError: (psycopg2.OperationalError) connection to server at "185.50.38.163", port 32112 failed: Connection timed out (0x0000274C/10060)
	Is the server running on that host and accepting TCP/IP connections?

(Background on this error at: https://sqlalche.me/e/20/e3q8)

## 6. Compare latency

Numbers or it did not happen.

In [ ]:
dashboard_sql = f'''
select neighbourhood, num_listings, avg_price, median_price,
       total_reviews, reviews_per_listing, availability_30_rate, availability_365_rate
from "{STUDENT_SCHEMA}".mv_airbnb_neighbourhood_summary
order by num_listings desc;
'''
dashboard_df, dashboard_times = timed_read_sql(dashboard_sql, repeats=5)
perf = pd.DataFrame([
    {'query': 'baseline_direct_query', 'best_seconds': min(baseline_times), 'avg_seconds': sum(baseline_times)/len(baseline_times)},
    {'query': 'materialized_view_read', 'best_seconds': min(dashboard_times), 'avg_seconds': sum(dashboard_times)/len(dashboard_times)},
])
perf['speedup_vs_baseline_best'] = perf.loc[0, 'best_seconds'] / perf['best_seconds']
perf

## 7. Metabase dashboard

Open the shared Metabase URL and create:

```text
QBC12 HW01 - <your-github-username> - Airbnb Ops
```

Required cards:

1. listings by neighbourhood
2. average price by neighbourhood
3. review activity by neighbourhood
4. availability rate by neighbourhood
5. top neighbourhoods table

Screenshot path:

```text
screenshots/metabase_dashboard.png
```

In [ ]:
# TODO 7.1
# Write reports/hw01_b_sql_performance.md.
# Include schema, runtimes, speedup, what changed, and Metabase screenshot/link.

# Write your code here.

In [ ]:
for file in ['sql/01_baseline_neighbourhood_summary.sql','sql/02_create_materialized_view.sql','reports/baseline_explain_analyze.txt','reports/explain_notes.md','reports/hw01_b_sql_performance.md']:
    assert Path(file).exists(), f'Missing {file}'
assert len(dashboard_df) > 0
perf

## Commit

```bash
git add sql reports screenshots notebooks
git commit -m "HW01-B SQL performance and Metabase dashboard"
```